<h1>Introduction</h1>

This project explores whether it is possible to estimate the playing strength of two chess players, represented by their average Elo rating—using only the moves of a game written in Chess Algebraic Notation (CAN). Using a dataset of 20,000 Lichess games played in 2017, I built a machine learning pipeline in Python capable of parsing PGN files, extracting features from the moves, and training a predictive model to estimate the players’ average rating.

My best model achieved a training score of 0.7285 and a testing score of 0.3526. While the test score shows the difficulty of the task, it also suggests that meaningful rating-related patterns do exist in move sequences. With more computational resources, a larger dataset, and deeper feature engineering, this approach could be extended much further.

<h1>Methods and Results</h1>

**Dataset and Preprocessing**

I began by loading 20,000 PGN games using the python-chess library. For each game, I extracted:

- Player metadata (Elo, names, results)

- Opening information (ECO code, named opening)

- Time control and termination type

- The full sequence of moves in SAN

- Total move count

A custom parser converted all games into a structured DataFrame, and only entries with valid integer Elo values were retained. I computed the mean Elo of the white and black player to use as the prediction target.

**Feature Engineering**

A variety of feature-engineering approaches were tested:

- Text-based features of moves using TF-IDF on SAN move sequences

- Opening-only features (subset of moves through TF-IDF)

-Engine-inspired handcrafted features, such as:

        -Number of blunders (approximate heuristic)

        -Number of queen moves in the opening

        -Number of checks delivered

        -Overall move count

Interestingly, most handcrafted features produced low importance and encouraged overfitting. I eventually discovered that restricting the model to opening moves produced significantly better generalization. This aligns with the intuition that openings are highly structured and strongly correlated with player strength.

**Modeling Approach**

The final model is an ensemble based on a VotingRegressor consisting of:

- Random Forest Regressor

- Gradient Boosting Regressor

Input features were processed using:

- One-Hot Encoding for time control

- TF-IDF Vectorization of the extracted opening-move text

**Final Model Performance**

Metric	Score
Training R²	0.7285
Testing R²	0.3526

The gap between training and test performance indicates notable overfitting—likely due to limited dataset size and the complexity of the input space.

**Key Findings**

Opening-only models outperformed full-game models.
Later parts of a chess game are much more chaotic and vary heavily between players of all strengths.
Feature engineering beyond openings added noise.
Naive blunder counts and heuristics did not meaningfully improve predictions.
Ratings are inherently difficult to infer from a single game.
Even strong players can have bad games, and lower-rated players can occasionally play well, introducing unavoidable variance.

In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn import *
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.dummy import DummyClassifier
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
import chess.pgn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import numpy as np
from scipy.stats import uniform
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor
import joblib

In [7]:
with open("games.pgn") as pgn:
    rows = []
    max_games = 20000
    
    for i in range(max_games):
        game = chess.pgn.read_game(pgn)
        if game is None:
            break
            
        headers = game.headers
        row = {
            "event": headers.get("Event"),
            "site": headers.get("Site"),
            "date": headers.get("Date"),
            "white": headers.get("White"),
            "black": headers.get("Black"),
            "result": headers.get("Result"),
            "white_elo": headers.get("WhiteElo"),
            "black_elo": headers.get("BlackElo"),
            "opening": headers.get("Opening"),
            "eco": headers.get("ECO"),
            "time_control": headers.get("TimeControl"),
            "termination": headers.get("Termination"),
        }

        board = game.board()
        moves_list = []
        
        for move in game.mainline_moves():
            algebraic_move = board.san(move)
            moves_list.append(algebraic_move)
            board.push(move)
        
        row["moves"] = " ".join(moves_list)
        row["move_count"] = len(moves_list)

        rows.append(row)

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} games")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)  


df.info()

Loaded 20000 games
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   event         20000 non-null  object
 1   site          20000 non-null  object
 2   date          20000 non-null  object
 3   white         20000 non-null  object
 4   black         20000 non-null  object
 5   result        20000 non-null  object
 6   white_elo     20000 non-null  object
 7   black_elo     20000 non-null  object
 8   opening       20000 non-null  object
 9   eco           20000 non-null  object
 10  time_control  20000 non-null  object
 11  termination   20000 non-null  object
 12  moves         20000 non-null  object
 13  move_count    20000 non-null  int64 
dtypes: int64(1), object(13)
memory usage: 2.1+ MB


In [8]:
data = df[['white_elo', 'black_elo', 'time_control', 'moves']].copy()
data = data[(data['white_elo'].str.isdigit()) & (data['black_elo'].str.isdigit())]
data['mean_elo'] = (data['white_elo'].astype(int) + data['black_elo'].astype(int)) / 2
data.head(2)

,white_elo,black_elo,time_control,moves,mean_elo
0,1833,1799,300+0,e4 d5 exd5 Qxd5 Nc3 Qd8 d4 Bf5 g4 Bg6 h4 h6 f4 e6 Nf3 c5 Ne5 a6 Nxg6 fxg6 Bc4 cxd4 Ne4 b5 Bxe6 N...,1816.0
1,1708,1556,300+3,d4 g6 c4 Bg7 Nc3 d6 Bf4 Nf6 h3 O-O Nf3 Re8 e3 a6 Bd3 c6 O-O b5 b3 b4 Ne4 Nxe4 Bxe4 e5 dxe5 dxe5 ...,1632.0


In [ ]:
data['mean_elo'].describe()

count    20000.000000
mean      1649.557600
std        278.143577
min        854.000000
25%       1461.875000
50%       1649.750000
75%       1836.000000
max       2697.500000
Name: mean_elo, dtype: float64

In [9]:
data['opening_only'] = data['moves'].apply(lambda x: ' '.join(x.split()[:15]))
data.head(2)

,white_elo,black_elo,time_control,moves,mean_elo,opening_only
0,1833,1799,300+0,e4 d5 exd5 Qxd5 Nc3 Qd8 d4 Bf5 g4 Bg6 h4 h6 f4 e6 Nf3 c5 Ne5 a6 Nxg6 fxg6 Bc4 cxd4 Ne4 b5 Bxe6 N...,1816.0,e4 d5 exd5 Qxd5 Nc3 Qd8 d4 Bf5 g4 Bg6 h4 h6 f4 e6 Nf3
1,1708,1556,300+3,d4 g6 c4 Bg7 Nc3 d6 Bf4 Nf6 h3 O-O Nf3 Re8 e3 a6 Bd3 c6 O-O b5 b3 b4 Ne4 Nxe4 Bxe4 e5 dxe5 dxe5 ...,1632.0,d4 g6 c4 Bg7 Nc3 d6 Bf4 Nf6 h3 O-O Nf3 Re8 e3 a6 Bd3


In [10]:
X_opening = data[['time_control', 'opening_only']]
X_train_opening, X_test_opening, y_train_opening, y_test_opening = train_test_split(
    X_opening, data['mean_elo'], test_size=0.2, random_state=42
)

In [62]:
opening_preprocessor = ColumnTransformer(
    transformers=[
        ('time_control', OneHotEncoder(handle_unknown='ignore'), ['time_control']),
        ('opening', TfidfVectorizer(max_features=1000, ngram_range=(1,2)), 'opening_only')
    ]
)

In [63]:
ensemble_pipeline = Pipeline([
('preprocessor', opening_preprocessor),
('regressor', VotingRegressor([
('rf', RandomForestRegressor(n_estimators=200, random_state=42)),
('gb', GradientBoostingRegressor(n_estimators=200, random_state=42)),
]))
])

In [65]:

param_distributions = {
    'preprocessor__opening__max_features': [500, 1000, 1500, 2000],
    'preprocessor__opening__ngram_range': [(1,1), (1,2)],
    'preprocessor__opening__min_df': [1, 2, 3],
    'regressor__rf__n_estimators': [100, 150, 200],
    'regressor__rf__max_depth': [20, 30, None],
    'regressor__rf__min_samples_split': [2, 5],
    'regressor__gb__n_estimators': [100, 150, 200],
    'regressor__gb__learning_rate': [0.05, 0.1, 0.15],
    'regressor__gb__max_depth': [3, 5, 7]
}

hyperopt_search = RandomizedSearchCV(
    ensemble_pipeline,
    param_distributions,
    n_iter=20,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

print("Starting hyperparameter optimization...")
hyperopt_search.fit(X_train_opening, y_train_opening)

print("Best parameters:")
print(hyperopt_search.best_params_)
print(f"Best CV score: {hyperopt_search.best_score_:.4f}")

best_model = hyperopt_search.best_estimator_
final_score = best_model.score(X_test_opening, y_test_opening)
print(f"Final test score: {final_score:.4f}")

Starting hyperparameter optimization...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters:
{'regressor__rf__n_estimators': 200, 'regressor__rf__min_samples_split': 2, 'regressor__rf__max_depth': 30, 'regressor__gb__n_estimators': 150, 'regressor__gb__max_depth': 7, 'regressor__gb__learning_rate': 0.15, 'preprocessor__opening__ngram_range': (1, 2), 'preprocessor__opening__min_df': 3, 'preprocessor__opening__max_features': 2000}
Best CV score: 0.3326
Final test score: 0.3526


In [11]:

optimal_preprocessor = ColumnTransformer(
    transformers=[
        ('time_control', OneHotEncoder(handle_unknown='ignore'), ['time_control']),
        ('opening', TfidfVectorizer(
            max_features=2000, 
            ngram_range=(1, 2), 
            min_df=3
        ), 'opening_only')
    ]
)

optimal_ensemble_pipeline = Pipeline([
    ('preprocessor', optimal_preprocessor),
    ('regressor', VotingRegressor([
        ('rf', RandomForestRegressor(
            n_estimators=200, 
            min_samples_split=2, 
            max_depth=30, 
            random_state=42
        )),
        ('gb', GradientBoostingRegressor(
            n_estimators=150, 
            max_depth=7, 
            learning_rate=0.15, 
            random_state=42
        )),
    ]))
])


print("Training optimal model with best hyperparameters...")
optimal_ensemble_pipeline.fit(X_train_opening, y_train_opening)


train_score = optimal_ensemble_pipeline.score(X_train_opening, y_train_opening)
test_score = optimal_ensemble_pipeline.score(X_test_opening, y_test_opening)

print(f"Optimal model train score: {train_score:.4f}")
print(f"Optimal model test score: {test_score:.4f}")


joblib.dump(optimal_ensemble_pipeline, 'chess_rating_optimal_model.pkl')

optimal_model_info = {
    'best_params': {
        'regressor__rf__n_estimators': 200,
        'regressor__rf__min_samples_split': 2,
        'regressor__rf__max_depth': 30,
        'regressor__gb__n_estimators': 150,
        'regressor__gb__max_depth': 7,
        'regressor__gb__learning_rate': 0.15,
        'preprocessor__opening__ngram_range': (1, 2),
        'preprocessor__opening__min_df': 3,
        'preprocessor__opening__max_features': 2000
    },
    'train_score': train_score,
    'test_score': test_score,
    'model_type': 'Optimal_VotingRegressor_RF_GB_Ensemble'
}
joblib.dump(optimal_model_info, 'chess_optimal_model_info.pkl')

print("Optimal model saved successfully!")

Training optimal model with best hyperparameters...
Optimal model train score: 0.7285
Optimal model test score: 0.3526
Optimal model saved successfully!


In [24]:
prediction = optimal_ensemble_pipeline.predict(X_test_opening.iloc[0:10000])
print(prediction.mean())
print(np.std(prediction))

1652.060362582433
144.3847573876347
